# Mini Project 2 — Part 2: Data Visualization

This notebook processes `jmurph91_project_summary.csv` from Part 1.

It:
1. Groups commit activity by month.
2. Includes zero-commit months between the first and last active month.
3. Creates one semicolon-separated monthly CSV for each project.
4. Creates one 300-DPI line plot for each project.
5. Calculates the longest inactivity gap.
6. Calculates the number of commits after the longest gap.
7. Counts inactivity gaps of at least three months.
8. Classifies the overall activity pattern for each project.
9. Creates the Part 2 project statistics table.

In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

NETID = "jmurph91"

INPUT_FILE = Path(f"{NETID}_project_summary.csv")
OUTPUT_DIR = Path("part2_outputs")

OUTPUT_DIR.mkdir(exist_ok=True)

print("Input file:", INPUT_FILE)
print("Output directory:", OUTPUT_DIR.resolve())

Input file: jmurph91_project_summary.csv
Output directory: /home/jmurph91/archeology/MiniProject2/MiniProject2/part2_outputs


In [19]:
df = pd.read_csv(INPUT_FILE, sep=";")

expected_columns = [
    "project_wocid",
    "commit_sha1",
    "author",
    "time",
    "commit message"
]

assert list(df.columns) == expected_columns, (
    f"Unexpected columns.\n"
    f"Expected: {expected_columns}\n"
    f"Found: {list(df.columns)}"
)

df["datetime"] = pd.to_datetime(
    df["time"],
    unit="s",
    errors="coerce"
)

assert df["datetime"].notna().all()

df["Month"] = df["datetime"].dt.to_period("M")

projects = sorted(df["project_wocid"].unique())

print(f"Total commits: {len(df):,}")
print(f"Number of projects: {len(projects)}")

for project in projects:
    print(" -", project)

Total commits: 50,009
Number of projects: 10
 - angel-ml_angel
 - bmclab_bmc
 - brial_brial
 - comit-network_xmr-btc-swap
 - easystats_insight
 - jasp-stats_jasp-desktop
 - juliarobotics_rigidbodydynamics.jl
 - mmorise_world
 - samreay_chainconsumer
 - systempiper_systempipeshiny


In [20]:
def create_monthly_timeseries(project_df):
    """
    Create a monthly commit time series.

    The resulting dataframe contains every month between
    the project's first and last active month, including
    months with zero commits.
    """

    monthly_counts = project_df.groupby("Month").size()

    first_month = monthly_counts.index.min()
    last_month = monthly_counts.index.max()

    all_months = pd.period_range(
        start=first_month,
        end=last_month,
        freq="M"
    )

    timeseries = pd.DataFrame({
        "Month": all_months.astype(str),
        "#Commits": monthly_counts
            .reindex(all_months, fill_value=0)
            .astype(int)
            .values
    })

    return timeseries

In [21]:
timeseries_data = {}

for project in projects:

    project_df = df[
        df["project_wocid"] == project
    ].copy()

    timeseries = create_monthly_timeseries(project_df)

    timeseries_data[project] = timeseries

    output_file = (
        OUTPUT_DIR /
        f"{NETID}_commits_timeseries_{project}.csv"
    )

    timeseries.to_csv(
        output_file,
        sep=";",
        index=False
    )

    print(
        f"{project}: "
        f"{len(timeseries)} months -> {output_file}"
    )

angel-ml_angel: 102 months -> part2_outputs/jmurph91_commits_timeseries_angel-ml_angel.csv
bmclab_bmc: 145 months -> part2_outputs/jmurph91_commits_timeseries_bmclab_bmc.csv
brial_brial: 236 months -> part2_outputs/jmurph91_commits_timeseries_brial_brial.csv
comit-network_xmr-btc-swap: 60 months -> part2_outputs/jmurph91_commits_timeseries_comit-network_xmr-btc-swap.csv
easystats_insight: 88 months -> part2_outputs/jmurph91_commits_timeseries_easystats_insight.csv
jasp-stats_jasp-desktop: 155 months -> part2_outputs/jmurph91_commits_timeseries_jasp-stats_jasp-desktop.csv
juliarobotics_rigidbodydynamics.jl: 108 months -> part2_outputs/jmurph91_commits_timeseries_juliarobotics_rigidbodydynamics.jl.csv
mmorise_world: 112 months -> part2_outputs/jmurph91_commits_timeseries_mmorise_world.csv
samreay_chainconsumer: 118 months -> part2_outputs/jmurph91_commits_timeseries_samreay_chainconsumer.csv
systempiper_systempipeshiny: 64 months -> part2_outputs/jmurph91_commits_timeseries_systempiper_s

In [22]:
for project in projects:

    print("\n" + "=" * 80)
    print(project)
    print("=" * 80)

    display(timeseries_data[project])


angel-ml_angel


,Month,#Commits
0,2017-05,6
1,2017-06,250
2,2017-07,167
3,2017-08,129
4,2017-09,127
...,...,...
97,2025-06,34
98,2025-07,5
99,2025-08,6
100,2025-09,4



bmclab_bmc


,Month,#Commits
0,2013-11,134
1,2013-12,92
2,2014-01,28
3,2014-02,9
4,2014-03,8
...,...,...
140,2025-07,0
141,2025-08,2
142,2025-09,1
143,2025-10,2



brial_brial


...

,Month,#Commits
0,2015-11,12
1,2015-12,3
2,2016-01,30
3,2016-02,27
4,2016-03,48
...,...,...
107,2024-10,4
108,2024-11,0
109,2024-12,0
110,2025-01,0



samreay_chainconsumer


,Month,#Commits
0,2016-07,62
1,2016-08,22
2,2016-09,22
3,2016-10,15
4,2016-11,0
...,...,...
113,2025-12,19
114,2026-01,0
115,2026-02,0
116,2026-03,32



systempiper_systempipeshiny


,Month,#Commits
0,2020-02,30
1,2020-03,21
2,2020-04,28
3,2020-05,19
4,2020-06,25
...,...,...
59,2025-01,0
60,2025-02,0
61,2025-03,0
62,2025-04,11


In [23]:
for project in projects:

    timeseries = timeseries_data[project]

    plt.figure(figsize=(10, 5))

    plt.plot(
        timeseries["Month"],
        timeseries["#Commits"],
        marker="o",
        linestyle="-"
    )

    plt.xlabel("Time (YYYY-MM)")
    plt.ylabel("Number of Commits")
    plt.title(project)

    plt.grid(True)
    plt.xticks(rotation=45)

    plt.tight_layout()

    output_file = (
        OUTPUT_DIR /
        f"{NETID}_timeseries_{project}.png"
    )

    plt.savefig(
        output_file,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    print("Saved:", output_file)

Saved: part2_outputs/jmurph91_timeseries_angel-ml_angel.png
Saved: part2_outputs/jmurph91_timeseries_bmclab_bmc.png
Saved: part2_outputs/jmurph91_timeseries_brial_brial.png
Saved: part2_outputs/jmurph91_timeseries_comit-network_xmr-btc-swap.png
Saved: part2_outputs/jmurph91_timeseries_easystats_insight.png
Saved: part2_outputs/jmurph91_timeseries_jasp-stats_jasp-desktop.png
Saved: part2_outputs/jmurph91_timeseries_juliarobotics_rigidbodydynamics.jl.png
Saved: part2_outputs/jmurph91_timeseries_mmorise_world.png
Saved: part2_outputs/jmurph91_timeseries_samreay_chainconsumer.png
Saved: part2_outputs/jmurph91_timeseries_systempiper_systempipeshiny.png


In [24]:
for project in projects:

    plot_file = (
        OUTPUT_DIR /
        f"{NETID}_timeseries_{project}.png"
    )

    assert plot_file.exists(), (
        f"Missing plot: {plot_file}"
    )

print("✓ All 10 required plots were generated.")

✓ All 10 required plots were generated.


In [25]:
def find_longest_gap(timeseries):
    """
    Find the longest continuous sequence of months
    containing zero commits.

    Returns:
        start_month
        end_month
        length
    """

    zero_mask = timeseries["#Commits"].eq(0)

    longest_length = 0
    longest_start = None
    longest_end = None

    current_start = None
    current_length = 0

    for i, is_zero in enumerate(zero_mask):

        if is_zero:

            if current_length == 0:
                current_start = timeseries.iloc[i]["Month"]

            current_length += 1

            if current_length > longest_length:
                longest_length = current_length
                longest_start = current_start
                longest_end = timeseries.iloc[i]["Month"]

        else:
            current_length = 0
            current_start = None

    return (
        longest_start,
        longest_end,
        longest_length
    )

In [26]:
def count_long_gaps(timeseries, minimum_length=3):
    """
    Count continuous zero-commit periods whose length
    is at least minimum_length months.
    """

    zero_mask = timeseries["#Commits"].eq(0)

    gap_count = 0
    current_length = 0

    for is_zero in zero_mask:

        if is_zero:
            current_length += 1

        else:
            if current_length >= minimum_length:
                gap_count += 1

            current_length = 0

    # Handle a gap ending at the final month.
    if current_length >= minimum_length:
        gap_count += 1

    return gap_count

In [27]:
def commits_after_longest_gap(timeseries):
    """
    Return the number of commits after the end of
    the longest inactivity gap.

    If there is no inactivity gap, return 0.
    """

    start, end, length = find_longest_gap(timeseries)

    if length == 0:
        return 0

    after_gap = timeseries[
        timeseries["Month"] > end
    ]

    return int(after_gap["#Commits"].sum())

In [28]:
def classify_activity_pattern(timeseries):
    """
    Classify the overall activity trajectory.

    Possible classifications:
    steady, rising, declining, U-shaped, cyclical, irregular
    """

    values = timeseries["#Commits"].to_numpy(dtype=float)

    if len(values) < 4:
        return "irregular"

    # Divide the timeline into thirds.
    thirds = np.array_split(values, 3)

    first_mean = np.mean(thirds[0])
    middle_mean = np.mean(thirds[1])
    last_mean = np.mean(thirds[2])

    overall_mean = np.mean(values)

    # Measure variability.
    if overall_mean > 0:
        coefficient_variation = (
            np.std(values) / overall_mean
        )
    else:
        coefficient_variation = 0

    # Linear trend.
    x = np.arange(len(values))

    if np.std(values) > 0:
        slope = np.polyfit(x, values, 1)[0]
        normalized_slope = (
            slope * len(values) / np.mean(values)
        )
    else:
        normalized_slope = 0

    # U-shaped: activity decreases and then increases.
    if (
        middle_mean < first_mean * 0.65
        and middle_mean < last_mean * 0.65
        and last_mean > middle_mean * 1.5
    ):
        return "U-shaped"

    # Rising.
    if normalized_slope > 0.35:
        return "rising"

    # Declining.
    if normalized_slope < -0.35:
        return "declining"

    # Cyclical: repeated variation across the timeline.
    if coefficient_variation > 1.0:
        return "cyclical"

    # Irregular: high variation without a strong trend.
    if coefficient_variation > 0.75:
        return "irregular"

    return "steady"

In [29]:
part2_records = []

for project in projects:

    project_df = df[
        df["project_wocid"] == project
    ].copy()

    timeseries = timeseries_data[project]

    gap_start, gap_end, gap_length = (
        find_longest_gap(timeseries)
    )

    commits_after_gap = (
        commits_after_longest_gap(timeseries)
    )

    number_of_gaps = (
        count_long_gaps(timeseries)
    )

    activity_pattern = (
        classify_activity_pattern(timeseries)
    )

    part2_records.append({
        "project": project,

        "number of commits":
            len(project_df),

        "number of authors":
            project_df["author"].nunique(),

        "max time":
            int(project_df["time"].max()),

        "min time":
            int(project_df["time"].min()),

        "LongestGapStart":
            gap_start if gap_start is not None else "N/A",

        "LongestGapEnd":
            gap_end if gap_end is not None else "N/A",

        "LongestGapLength":
            gap_length,

        "NumCommitsAfterLastGap":
            commits_after_gap,

        "NumberOfGapsInTimeline":
            number_of_gaps,

        "ActivityPattern":
            activity_pattern
    })

part2_stats = pd.DataFrame(part2_records)

part2_stats

,project,number of commits,number of authors,max time,min time,LongestGapStart,LongestGapEnd,LongestGapLength,NumCommitsAfterLastGap,NumberOfGapsInTimeline,ActivityPattern
0,angel-ml_angel,3418,147,1760616761,1493817527,2024-02,2025-05,16,54,2,declining
1,bmclab_bmc,2083,30,1762230919,1383617973,2016-12,2017-03,4,1372,4,declining
2,brial_brial,2999,34,1759360722,1141893962,2013-02,2015-04,27,186,14,declining
3,comit-network_xmr-btc-swap,4980,68,1755593438,1601272508,N/A,N/A,0,0,0,declining
4,easystats_insight,9905,86,1776168259,1548793879,N/A,N/A,0,0,0,steady
5,jasp-stats_jasp-desktop,22023,242,1777093856,1370113512,2013-11,2013-11,1,21975,0,rising
6,juliarobotics_rigidbodydynamics.jl,2984,23,1740965887,1460686712,2023-07,2024-09,15,27,4,declining
7,mmorise_world,348,46,1740126621,1447574109,2022-03,2023-05,15,11,9,declining
8,samreay_chainconsumer,634,18,1775513079,1468796482,2022-10,2023-09,12,242,7,U-shaped
9,systempiper_systempipeshiny,635,20,1747714763,1582064135,2023-11,2024-03,5,16,6,declining


In [30]:
display(
    part2_stats[
        [
            "project",
            "number of commits",
            "number of authors",
            "LongestGapStart",
            "LongestGapEnd",
            "LongestGapLength",
            "NumCommitsAfterLastGap",
            "NumberOfGapsInTimeline",
            "ActivityPattern"
        ]
    ]
)

,project,number of commits,number of authors,LongestGapStart,LongestGapEnd,LongestGapLength,NumCommitsAfterLastGap,NumberOfGapsInTimeline,ActivityPattern
0,angel-ml_angel,3418,147,2024-02,2025-05,16,54,2,declining
1,bmclab_bmc,2083,30,2016-12,2017-03,4,1372,4,declining
2,brial_brial,2999,34,2013-02,2015-04,27,186,14,declining
3,comit-network_xmr-btc-swap,4980,68,N/A,N/A,0,0,0,declining
4,easystats_insight,9905,86,N/A,N/A,0,0,0,steady
5,jasp-stats_jasp-desktop,22023,242,2013-11,2013-11,1,21975,0,rising
6,juliarobotics_rigidbodydynamics.jl,2984,23,2023-07,2024-09,15,27,4,declining
7,mmorise_world,348,46,2022-03,2023-05,15,11,9,declining
8,samreay_chainconsumer,634,18,2022-10,2023-09,12,242,7,U-shaped
9,systempiper_systempipeshiny,635,20,2023-11,2024-03,5,16,6,declining


In [31]:
print("=" * 90)
print("INTERPRETATION")
print("=" * 90)

for _, row in part2_stats.iterrows():

    print(f"\n{row['project']}")
    print("-" * len(row["project"]))

    if row["LongestGapLength"] == 0:
        print(
            "No inactivity gap occurred because the project had "
            "continuous monthly activity across its timeline."
        )
    else:
        print(
            f"The longest inactivity gap lasted "
            f"{row['LongestGapLength']} month(s), "
            f"from {row['LongestGapStart']} through "
            f"{row['LongestGapEnd']}."
        )

    print(
        f"The project had "
        f"{row['NumberOfGapsInTimeline']} inactivity gap(s) "
        f"lasting at least three months."
    )

    print(
        f"There were {row['NumCommitsAfterLastGap']} "
        f"commits after the longest gap."
    )

    print(
        f"The overall activity pattern was classified as "
        f"{row['ActivityPattern']}."
    )

INTERPRETATION

angel-ml_angel
--------------
The longest inactivity gap lasted 16 month(s), from 2024-02 through 2025-05.
The project had 2 inactivity gap(s) lasting at least three months.
There were 54 commits after the longest gap.
The overall activity pattern was classified as declining.

bmclab_bmc
----------
The longest inactivity gap lasted 4 month(s), from 2016-12 through 2017-03.
The project had 4 inactivity gap(s) lasting at least three months.
There were 1372 commits after the longest gap.
The overall activity pattern was classified as declining.

brial_brial
-----------
The longest inactivity gap lasted 27 month(s), from 2013-02 through 2015-04.
The project had 14 inactivity gap(s) lasting at least three months.
There were 186 commits after the longest gap.
The overall activity pattern was classified as declining.

comit-network_xmr-btc-swap
--------------------------
No inactivity gap occurred because the project had continuous monthly activity across its timeline.
The pro

In [32]:
PART2_STATS_FILE = (
    OUTPUT_DIR /
    f"{NETID}_part2_project_stats.csv"
)

part2_stats.to_csv(
    PART2_STATS_FILE,
    sep=";",
    index=False
)

print("Saved:", PART2_STATS_FILE)

Saved: part2_outputs/jmurph91_part2_project_stats.csv


In [33]:
for project in projects:

    path = (
        OUTPUT_DIR /
        f"{NETID}_commits_timeseries_{project}.csv"
    )

    check = pd.read_csv(path, sep=";")

    assert list(check.columns) == [
        "Month",
        "#Commits"
    ]

    assert check["Month"].notna().all()
    assert check["#Commits"].notna().all()
    assert (check["#Commits"] >= 0).all()

print("✓ All 10 monthly CSV files have the required format:")
print("  Month;#Commits")

✓ All 10 monthly CSV files have the required format:
  Month;#Commits
